## Mục lục
- [ 1. Cài đặt](#cai-dat)
- [ 2. Test Network](#test-network)
  - [ 2.1 Bật Network](#up-network)
  - [ 2.2 Tạo Channel](#create-channel)
  - [ 2.3 Khởi tạo một chaincode trên channel](#start-chaincode)
  - [ 2.4 Tương tác với network](#interact-network)
- [ 3. Triển khai Smart Contract lên Channel](#deploy-smart-contract)
  - [ 3.1 Khởi động Network và tạo Channel](#up-network-2)
  - [ 3.2 Chaincode as a Service (CCAAS) - Đóng gói Chaincode](#ccaas)
  - [ 3.3 Cài đặt Chaincode Package](#install-chaincode)
  - [ 3.4 Phê duyệt Chaincode Definition](#approve-chaincode)
  - [ 3.5 Đệ trình Chaincode Definition lên Channel](#commit-chaincode)
  - [ 3.6 Start Docker Container](#start-docker)
  - [ 3.7 Invoke chaincode](#invoke-chaincode)
  - [ 3.8 Sử dụng Application](#app)
  - [ 3.9 Upgrade một Smart Contract](#updrage-chaincode)
  - [ 3.10 Dọn dẹp](#clean-up)


Khi mở DevOps HyperLedger trên Fit-Lab, ta sẽ ở thư mục root, ví dụ `root@hyper-2001102857:~#`. Ta cần chuyển đến thư mục `hyper` để thực hiện các tác vụ. Sử dụng lệnh sau để chuyển thư mục

In [ ]:
su -l hyper

Ta sẽ chuyển đến thư mục có dạng `hyper@hyper-2001102857:~$`

<a id='cai-dat'></a>
## 1. Cài đặt

Tải `go` version 1.24.0

In [ ]:
wget https://dl.google.com/go/go1.24.0.linux-amd64.tar.gz

Remove phiên bản GO hiện có

In [ ]:
rm -rf /home/hyper/go

Cài đặt GO mới

In [ ]:
tar -C /home/hyper -xzf go1.24.0.linux-amd64.tar.gz

Kiểm tra kết quả cài đặt

In [ ]:
go version

Hệ thống nên trả về thông tin của phiên bản GO được cài đặt. Nếu gặp lỗi `-bash: go: command not found`, ta cần đăng ký

In [ ]:
export PATH=$PATH:/home/hyper/go/bin

source ~/.bashrc

Về sau, trước khi làm việc với DevOps Hyperledger, ta cần làm lại bước kiểm tra xem command Go có hoạt động hay không.

Xóa file go1.24.0.linux-amd64.tar.gz để giải phóng bộ nhớ

In [ ]:
rm go1.24.0.linux-amd64.tar.gz

Thiết lập các biến môi trường cho GO.

Chú ý đường dẫn GOPATH trỏ tới thư mục fabric-samples là thư mục chứa project (mẫu) mà chúng ta sẽ làm việc. Ở lần chạy đầu tiên, thư mục này chưa có (sẽ được cài đặt ngay dưới đây). Nếu ta làm việc với một project khác thì ta cần thiết lập GOPATH trỏ tới thư mục chứa project đó (ta cần chuẩn bị file `go.mod` tương tự như trong project fabric-samples này).

In [ ]:
go env -w GOPATH=/home/hyper/go/src/fabric-samples/

go env -w GO111MODULE=on

go env -w GOMODCACHE=/home/hyper/go/pkg/mod

Di chuyển đến thư mục go/src, tải về và cài đặt fabric-samples

In [ ]:
cd go/src

curl -sSLO https://raw.githubusercontent.com/hyperledger/fabric/main/scripts/install-fabric.sh && chmod +x install-fabric.sh

./install-fabric.sh docker samples binary

Về thư mục gốc và cài đặt `nvm` mới hơn để chạy ứng dụng NodeJS cho Fabric

In [ ]:
cd ~

curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash

source ~/.bashrc

nvm install 20

nvm use 20

<a id='test-network'></a>
## 2. Test Network

<a id='up-network'></a>
### 2.1 Bật Network

Di chuyển đến thư mục `fabric-samples/test-network`

In [ ]:
cd fabric-samples/test-network

Chạy lệnh sau để gỡ bỏ tất cả tàn dư của những lần chạy trước đó

In [ ]:
docker stop $(docker ps -q)

./network.sh down

Chạy lệnh sau để tạo một Network gồm `2 peer node` và `1 ordering node`.

In [ ]:
./network.sh up

<a id='create-channel'></a>
### 2.2 Tạo Channel

Ta có thể sử dụng `network.sh` script để tạo một channel giữa Org1 và Org2 và cho các peer của chúng tham gia vào channel.

Lệnh sau sẽ tạo một channel có tên mặc định là `mychannel`. Nếu lệnh này thành công, chúng ta sẽ thấy có thông báo "Channel 'mychannel' joined."

In [ ]:
./network.sh createChannel -ca

Chúng ta cũng có thể sử dụng channel flag để tạo một channel với tên tùy chỉnh. Ví dụ, muốn tạo một channel có tên "mychannel1", chúng ta sẽ chạy lệnh sau

In [ ]:
./network.sh createChannel -c channel1 -ca

Chúng ta có thể tạo nhiều channel sử dụng lệnh trên.

<a id='start-chaincode'></a>
### 2.3 Khởi tạo một chaincode trên channel

Sau khi đã tạo một channel, chúng ta có thể khởi tạo một chaincode trên channel (`mychannel`) sử dụng lệnh sau

In [ ]:
./network.sh deployCCAAS -ccn basic -ccp ../asset-transfer-basic/chaincode-external

Lệnh con `deployCC` sẽ cài đặt chaincode **asset-transfer (basic)** trên `peer0.org1.example.com` và `peer0.org2.example.com`, sau đó sẽ triển khai chaincode trên channel được nêu rõ sử dụng channel flag (nếu không có channel nào được chỉ định, `mychannel` sẽ được sử dụng). Trong lệnh trên, chúng ta sử dụng language flag `-ccl` để cài đặt các phụ thuộc cuar chaincode dùng phiên bản ngôn ngữ Go.

Ví dụ khi ta muốn deploy `channel1` được tạo bằng lệnh `./network.sh createChannel -c channel1` bên trên

In [ ]:
./network.sh deployCCAAS -c channel1 -ccn basic -ccp ../asset-transfer-basic/chaincode-external

<a id='interact-network'></a>
### 2.4 Tương tác với network

Sau khi đã khởi động test network, chúng ta có thể sử dụng `peer` CLI để tương tác với network. `peer` CLI cho phép chúng ta gọi ra những smart contract đã được triển khai, hoặc cài đặt và triển khai những smart contract mới.

Từ thư mục `test-network`, chạy lệnh sau để thêm các `peer` binaries (nằm trong thư mục `fabric-samples/bin`) vào CLI path

In [ ]:
export PATH=${PWD}/../bin:$PATH

Ta cũng cần thiết lập `FABRIC_CFG_PATH` để trỏ đến file `core.yaml` trong `fabric-samples` repository

In [ ]:
export FABRIC_CFG_PATH=$PWD/../config/

Tiếp theo ta cần thiết lập các biến môi trường để cho phép ta vận hành `peer` CLI như là Org1

In [ ]:
export CORE_PEER_TLS_ENABLED=true

export CORE_PEER_LOCALMSPID="Org1MSP"

export CORE_PEER_TLS_ROOTCERT_FILE=${PWD}/organizations/peerOrganizations/org1.example.com/peers/peer0.org1.example.com/tls/ca.crt

export CORE_PEER_MSPCONFIGPATH=${PWD}/organizations/peerOrganizations/org1.example.com/users/Admin@org1.example.com/msp

export CORE_PEER_ADDRESS=localhost:7051

Các biến môi trường `CORE_PEER_TLS_ROOTCERT_FILE` và `CORE_PEER_MSPCONFIGPATH` trỏ đến Org1 crypto material trong thư mục `organizations`.

Sau khi đã cài đặt và khởi động **asset-transfer (basic)** (dùng lệnh `./network.sh deployCC -ccl go` bên trên), ta có thể gọi hàm `InitLedger` của chaincode để thiết lập danh sách khởi đầu của **asset** trên **ledger**. Chạy lệnh sau để khởi tạo ledger với các asset

In [ ]:
peer chaincode invoke -o localhost:7050 --ordererTLSHostnameOverride orderer.example.com --tls --cafile "${PWD}/organizations/ordererOrganizations/example.com/orderers/orderer.example.com/msp/tlscacerts/tlsca.example.com-cert.pem" -C mychannel -n basic --peerAddresses localhost:7051 --tlsRootCertFiles "${PWD}/organizations/peerOrganizations/org1.example.com/peers/peer0.org1.example.com/tls/ca.crt" --peerAddresses localhost:9051 --tlsRootCertFiles "${PWD}/organizations/peerOrganizations/org2.example.com/peers/peer0.org2.example.com/tls/ca.crt" -c '{"function":"InitLedger","Args":[]}'

Nếu thành công, ta sẽ thấy output tương tự như sau

In [ ]:
2020-02-12 18:22:20.576 EST [chaincodeCmd] chaincodeInvokeOrQuery -> INFO 001 Chaincode invoke successful. result: status:200

Run application

In [ ]:
cd ../asset-transfer-basic/application-gateway-go

go run .

<a id='deploy-smart-contract'></a>
## 3. Triển khai Smart Contract lên Channel

<a id='up-network-2'></a>
### 3.1 Khởi động Network và tạo Channel

Vào thư mục `fabric-samples/test-network`

In [ ]:
cd fabric-samples/test-network

In [ ]:
docker stop $(docker ps -q)

./network.sh down

./network.sh up

./network.sh createChannel -ca

Ta thấy rằng có 3 docker container tương ứng với các peer node bao gồm
  * Tên: peer0.org1.example.com, vai trò **peer node start**, port **7051**
  * Tên: peer0.org2.example.com, vai trò **peer node start**, port **9051**
  * Tên: orderer.example.com, vai trò **orderer**, port **7050**

<a id='ccaas'></a>
### 3.2 Chaincode as a Service (CCAAS) - Đóng gói Chaincode

Sử dụng **Chaincode as a Service (CCAAS)** sẽ giúp tách biệt việc quản lý vòng đời chaincode khỏi Peer, chạy chaincode như một container độc lập hoặc một dịch vụ bên ngoài. Để chạy Chaincode-as-a-Service, chúng ta sẽ tự tạo ra một docker image để đóng gói mã nguồn Go thành một service. Khi đó
* **Peer** chỉ đóng vai trò là một client gửi yêu cầu gRPC.
* **Chaincode** chạy như một ứng dụng bình thường.

Docker file để build docker image có trong thư mục `fabric-samples/asset-transfer-basic/chaincode-external`. Chúng ta sẽ chỉ dẫn đến thư mục trên cho Dockerfile và source code cần thiết để build một docker image có tên `basic_ccaas_image` (**basic** cũng là tên mà ta dự định đặt cho chaincode), tag `latest` version, đồng thời chỉ dẫn chaincode lắng nghe port 9999.

In [ ]:
docker build -f ../asset-transfer-basic/chaincode-external/Dockerfile -t basic_ccaas_image:latest --build-arg CC_SERVER_PORT=9999 ../asset-transfer-basic/chaincode-external

Nếu trước đó chúng ta di chuyển đến thư mục `fabric-samples/asset-transfer-basic/chaincode-external` thì câu lệnh trên sẽ đơn giản hóa thành

In [ ]:
docker build -t basic_ccaas_image:latest --build-arg CC_SERVER_PORT=9999 .

Ta sẽ tạo một chaincode package có tên **basic**. Nó sẽ là một file zip **basic.tar.gz** có cấu trúc 2-layer được thiết lập bằng cách dùng các lệnh như bên dưới. Lưu ý, 9999 chính là `CC_SERVER_PORT` ta dùng trong Dockerfile khi build docker image bên trên. File package này chỉ đóng vai trò thiết lập kết nối, không chứa source code (nằm trong docker image mà ta build bên trên).

In [ ]:
cd ../asset-transfer-basic/chaincode-external

export address=peer0.org1.example.com_basic_ccaas:9999

export label=basic_1.0

mkdir -p tempdir/src

cat > "tempdir/src/connection.json" <<CONN_EOF
{
  "address": "${address}",
  "dial_timeout": "10s",
  "tls_required": false
}
CONN_EOF

mkdir -p "tempdir/pkg"

cat << METADATA-EOF > "tempdir/pkg/metadata.json"
{
    "type": "ccaas",
    "label": "$label"
}
METADATA-EOF

tar -C "tempdir/src" -czf "tempdir/pkg/code.tar.gz" .

tar -C "tempdir/pkg" -czf "basic.tar.gz" metadata.json code.tar.gz

rm -Rf "tempdir"

Để có thể làm việc với `peer`, ta cần thêm các `peer` binaries (nằm trong thư mục `fabric-samples/bin`) vào CLI path. Ta có thể quay lại thư mục `test-network`.

In [ ]:
cd ../../test-network

export PATH=${PWD}/../bin:$PATH

export FABRIC_CFG_PATH=$PWD/../config/

Để chắc chắn rằng ta có thể sủ dụng `peer` CLI, ta kiểm tra phiên bản của các binaries này, ta cần phiên bản 2.0.0 hoặc mới hơn

In [ ]:
peer version

Để xem ID của chaincode package, ta có thể dùng lệnh

In [ ]:
peer lifecycle chaincode calculatepackageid ../asset-transfer-basic/chaincode-external/basic.tar.gz

Kết quả sẽ tương tự như sau

In [ ]:
basic_1.0:9149eb670ec38539b2ce8117c5fa758cd674fd27b77a8a6b9100c528c4209be3

Nếu muốn gán ID này cho một biến `PACKAGE_ID`, ta dùng lệnh

In [ ]:
PACKAGE_ID=$(peer lifecycle chaincode calculatepackageid ../asset-transfer-basic/chaincode-external/basic.tar.gz)

Khi cần truy xuất package ID, chúng ta sẽ gọi `$PACKAGE_ID`.

In [ ]:
echo $PACKAGE_ID

Chúng ta đã tạo xong chaincode package. Bây giờ chúng ta sẽ cài đặt chaincode lên các peers của test network.

<a id='install-chaincode'></a>
### 3.3 Cài đặt Chaincode Package

Sau khi đã đóng gói smart contract **asset-transfer (basic)**, ta có thể cài đặt chaincode lên các **peers**. Chaincode cần được cài đặt lên tất cả các peers sẽ phê duyệt giao dịch. Trong ví dụ này, chúng ta sẽ thiết lập **chính sách phê duyệt** yêu cầu sự phê duyệt từ cả hai tổ chức Org1 và Org2, cho nên ta cần cài đặt chaincode lên các peers được vận hành bởi cả hai tổ chức này
  * peer0.org1.example.com
  * peer0.org2.example.com

Đầu tiên ta sẽ cài đặt chaincode lên peer của Org1. Thiết lập các biến môi trường để vận hành peer CLI như thể admin user của Org1. Biến `CORE_PEER_ADDRESS` sẽ được thiết lập để trỏ đến peer Org1 là `peer0.org1.example.com`, port 7051.

In [ ]:
export CORE_PEER_TLS_ENABLED=true

export CORE_PEER_LOCALMSPID="Org1MSP"

export CORE_PEER_TLS_ROOTCERT_FILE=${PWD}/organizations/peerOrganizations/org1.example.com/peers/peer0.org1.example.com/tls/ca.crt

export CORE_PEER_MSPCONFIGPATH=${PWD}/organizations/peerOrganizations/org1.example.com/users/Admin@org1.example.com/msp

export CORE_PEER_ADDRESS=localhost:7051

Sử dụng lệnh `peer lifecycle chaincode install` để cài đặt chaincode lên peer

In [ ]:
peer lifecycle chaincode install ../asset-transfer-basic/chaincode-external/basic.tar.gz

Nếu lệnh trên thành công, peer sẽ sinh ra và trả về **package ID** sẽ được dùng để phê duyệt chaincode ở bước kế tiếp. Ta sẽ thấy output tương tự như sau

In [ ]:
2026-04-05 15:01:58.759 UTC 0001 INFO [cli.lifecycle.chaincode] submitInstallProposal -> Installed remotely: response:<status:200 payload:"\nJbasic_1.0:9149eb670ec38539b2ce8117c5fa758cd674fd27b77a8a6b9100c528c4209be3\022\tbasic_1.0" > 
2026-04-05 15:01:58.759 UTC 0002 INFO [cli.lifecycle.chaincode] submitInstallProposal -> Chaincode code package identifier: basic_1.0:9149eb670ec38539b2ce8117c5fa758cd674fd27b77a8a6b9100c528c4209be3

Tiếp theo ta sẽ cài đặt chaincode lên Org2 peer. Thiết lập các biến môi trường để vận hành như thể Org2 admin và trỏ đến Org2 peer là `peer0.org2.example.com`, port 9051.

In [ ]:
export CORE_PEER_LOCALMSPID="Org2MSP"

export CORE_PEER_TLS_ROOTCERT_FILE=${PWD}/organizations/peerOrganizations/org2.example.com/peers/peer0.org2.example.com/tls/ca.crt

export CORE_PEER_MSPCONFIGPATH=${PWD}/organizations/peerOrganizations/org2.example.com/users/Admin@org2.example.com/msp

export CORE_PEER_ADDRESS=localhost:9051

Cài đặt chaincode lên peer

In [ ]:
peer lifecycle chaincode install ../asset-transfer-basic/chaincode-external/basic.tar.gz

<a id='approve-chaincode'></a>
### 3.4 Phê duyệt Chaincode Definition

Sau khi đã cài đặt chaincode package, ta cần phê duyệt chaincode definition cho các tổ chức. Chaincode definition bao gồm các tham số quan trọng của chaincode như: name, version, chính sách phê duyệt.

Tập hợp các thành viên trên channel, những người cần phải phê duyệt một chaincode trước khi nó được triển khai, được quản lý bởi chính sách `/Channel/Application/LifecycleEndorsement`. Mặc định, chính sách này yêu cầu đa số thành viên trên channel phê duyệt một chaincode trước khi nó được sử dụng trên channel. Trong ví dụ này, chúng ta có hai tổ chức trên channel, và đa số của 2 cũng là 2, vì vậy chúng ta cần cả Org1 và Org2 đều phê duyệt chaincode definition của asset-transfer (basic).

Nếu một tổ chức đã cài đặt chaincode lên peer của mình, họ cần đưa package ID và chaincode definition mà họ phê duyệt. Package ID được sử dụng để liên kết chaincode đã được cài đặt trên một peer với một chaincode definition đã được phê duyệt, và cho phép tổ chức sử dụng chaincode để phê duyệt các giao dịch. Ta có thể tìm thông tin package ID bằng cách sử dụng lệnh `peer lifecycle chaincode queryinstalled` để truy vấn peer của mình. (Lưu ý rằng, ta vẫn đang ở trên peer Org2 là `peer0.org2.example.com`.)

In [ ]:
peer lifecycle chaincode queryinstalled

Package ID là kết hợp của **chaincode label** và một hash của chaincode binaries. Mọi peer sẽ sinh ra cùng một package ID. Ta sẽ thấy output tương tự như sau

In [ ]:
Installed chaincodes on peer:
Package ID: basic_1.0:9149eb670ec38539b2ce8117c5fa758cd674fd27b77a8a6b9100c528c4209be3, Label: basic_1.0

Chúng ta sẽ sử dụng package ID khi chúng ta phê duyệt chaincode. Lưu ý rằng ta đã lưu nó trong biến môi trường `PACKAGE_ID` bên trên.

Bởi vì chúng ta đang ở vai trò của Org2 admin, chúng ta có thể phê duyệt chaincode definition của asset-transfer (basic) dưới danh nghĩa Org2. Chaincode được phê duyệt ở cấp độ tổ chức, cho nên lệnh phê duyệt chỉ cần nhắm đến một peer. Sự phê duyệt sẽ được phân đến các peer khác trong cùng tổ chức thông qua **gossip**. Phê duyệt chaincode definition sử dụng lệnh `peer lifecycle chaincode approveformyorg`

In [ ]:
peer lifecycle chaincode approveformyorg -o localhost:7050 --ordererTLSHostnameOverride orderer.example.com --channelID mychannel --name basic --version 1.0 --package-id $PACKAGE_ID --sequence 1 --tls --cafile "${PWD}/organizations/ordererOrganizations/example.com/orderers/orderer.example.com/msp/tlscacerts/tlsca.example.com-cert.pem"

Output có dạng như sau

In [ ]:
2026-04-05 15:08:33.402 UTC 0001 INFO [chaincodeCmd] ClientWait -> txid [4e55ac4d360d6799d7598c25a2a58efd49ce2b083a92153e87f985d1eb223ca1] committed with status (VALID) at localhost:9051

Lệnh trên sử dụng flag `--package-id` để đưa vào package ID trong chaincode definition. Tham số `--sequence` là một số nguyên để theo dõi số lần một chaincode được định nghĩa hoặc cập nhật. Ở đây, chaincode được triển khai lên channel lần đầu nên tham số này bằng 1. Khi chaincode asset-transfer (basic) này được upgraded, tham số này sẽ là 2.

Chúng ta cũng có thể cung cấp các tham số `--signature-policy` hoặc `--channel-config-policy` vào lệnh `approveformyorg` bên trên để chỉ rõ chính sách phê duyệt. Chính sách này quy định số peer (thuộc về các thành viên khác nhau của channel) cần thiết để phe duyệt một giao dịch. Ở đây ta không thiết lập chính sách phê duyệt, cho nên chaincode asset-transfer (basic) sẽ sử dụng chính sách mặc định là yêu cầu một giao dịch được phê duyệt bởi đa số thành viên channel khi giao dịch được đệ trình. Điều này có nghĩa là, nếu có tổ chức mới được thêm vào hoặc xóa khỏi channel, chính sách phê duyệt sẽ được tự động cập nhật để yêu cầu nhiều hơn hoặc ít hơn các phê duyệt. Trong ví dụ này chúng ta chỉ có hai tổ chức Org1 và Org2 nên sẽ cần sự phê duyệt từ cả hai.

Chúng ta cần phê duyệt một chaincode definition với định danh ở vai trò admin. Vì vậy, biến `CORE_PEER_MSPCONFIGPATH` cần trỏ tới thư mục MSP là nơi chứa định danh admin. Ta không thể phê duyệt một chaincode definition với một client user. Phê duyệt phải được đệ trình tới ordering service nơi sẽ xác minh chữ ký admin và phân bổ phê duyệt đó tới các peer trong cùng tổ chức.

Vừa nãy ta đã phê chuẩn chaincode definition dưới danh nghĩa Org2. Ta cần tiếp tục phê chuẩn chaincode dưới danh nghĩa Org1. Ta cần thiết lập các biến môi trường để vận hành như Org1 admin.

In [ ]:
export CORE_PEER_LOCALMSPID="Org1MSP"

export CORE_PEER_MSPCONFIGPATH=${PWD}/organizations/peerOrganizations/org1.example.com/users/Admin@org1.example.com/msp

export CORE_PEER_TLS_ROOTCERT_FILE=${PWD}/organizations/peerOrganizations/org1.example.com/peers/peer0.org1.example.com/tls/ca.crt

export CORE_PEER_ADDRESS=localhost:7051

Bây giờ ta có thể phê duyệt chaincode dưới danh nghĩa Org1.

In [ ]:
peer lifecycle chaincode approveformyorg -o localhost:7050 --ordererTLSHostnameOverride orderer.example.com --channelID mychannel --name basic --version 1.0 --package-id $PACKAGE_ID --sequence 1 --tls --cafile "${PWD}/organizations/ordererOrganizations/example.com/orderers/orderer.example.com/msp/tlscacerts/tlsca.example.com-cert.pem"

Output sẽ có dạng

In [ ]:
2026-04-05 15:09:40.187 UTC 0001 INFO [chaincodeCmd] ClientWait -> txid [83e48749103bb6f983af115ccb3d8db0e991ba34bf8a1eaf709d4455945e0c1d] committed with status (VALID) at localhost:7051

<a id='commit-chaincode'></a>
### 3.5 Commit Chaincode Definition lên Channel

Khi một chaincode definition đã được phê duyệt bởi đủ số lượng tổ chức cần thiết, một tổ chức có thể commit chaincode definition lên channel. 

Ta có thể sử dụng lệnh `peer lifecycle chaincode checkcommitreadiness` để kiểm tra xem các thành viên channel có phê duyệt cùng một chaincode definition hay không. Các **flags** sử dụng cho lệnh `checkcommitreadiness` phải giống với các flags đã sử dụng để phê duyệt chaincode. Tuy nhiên, ta không cần flag `--package-id`.

In [ ]:
peer lifecycle chaincode checkcommitreadiness --channelID mychannel --name basic --version 1.0 --sequence 1 --tls --cafile "${PWD}/organizations/ordererOrganizations/example.com/orderers/orderer.example.com/msp/tlscacerts/tlsca.example.com-cert.pem" --output json

Lệnh trên sẽ xuất ra một JSON map thể hiện xem một tổ chức có phê duyệt hay không các tham số được nêu ra ở lệnh `checkcommitreadiness`

In [ ]:
{
        "approvals": {
                "Org1MSP": true,
                "Org2MSP": true
        }
}

Vì cả hai tổ chức thành viên của channel đã phê duyệt cùng một bộ tham số, chaincode definition đã sẵn sàng được commit lên channel. Ta có thể sử dụng lênh `peer lifecycle chaincode commit` để commit chaincode. Lệnh commit cũng cần được đưa ra bởi một admin của tổ chức.

In [ ]:
peer lifecycle chaincode commit -o localhost:7050 --ordererTLSHostnameOverride orderer.example.com --channelID mychannel --name basic --version 1.0 --sequence 1 --tls --cafile "${PWD}/organizations/ordererOrganizations/example.com/orderers/orderer.example.com/msp/tlscacerts/tlsca.example.com-cert.pem" --peerAddresses localhost:7051 --tlsRootCertFiles "${PWD}/organizations/peerOrganizations/org1.example.com/peers/peer0.org1.example.com/tls/ca.crt" --peerAddresses localhost:9051 --tlsRootCertFiles "${PWD}/organizations/peerOrganizations/org2.example.com/peers/peer0.org2.example.com/tls/ca.crt"

Output sẽ có dạng tương tự như sau

In [ ]:
2026-04-05 15:10:46.339 UTC 0001 INFO [chaincodeCmd] ClientWait -> txid [f3ea1abea46fdffde57994ecc750b4b47d6b63ae0d444b9efcac0b68af9af6fc] committed with status (VALID) at localhost:9051
2026-04-05 15:10:46.339 UTC 0002 INFO [chaincodeCmd] ClientWait -> txid [f3ea1abea46fdffde57994ecc750b4b47d6b63ae0d444b9efcac0b68af9af6fc] committed with status (VALID) at localhost:7051

Giao dịch trên sử dụng flag `--peerAddresses` để trỏ đến `peer0.org1.example.com` của Org1 và `peer0.org2.example.com` của Org2. Giao dịch `commit` được đệ trình tới các **peers** tham gia channel để truy vấn chaincode definition đã được phê duyệt bởi tổ chức vận hành **peer**. Lệnh trên cần trỏ đến các peers sao cho đủ số lượng cần thiết các tổ chức để thỏa mãn chính sách triển khai chaincode. Bởi vì phép phê duyệt đã được phổ biến bên trong mỗi tổ chức, ta chỉ cần trỏ đến bất kỳ peer nào của mỗi thành viên channel.

Phê duyệt của chaincode definition được commit tới ordering service để thêm vào block và đưa lên channel. Các peers trên channel xác minh xem có đủ số lượng cần thiết các thành viên đã phê duyệt chaincode hay không. Lệnh `peer lifecycle chaincode commit` sẽ đợi sự xác minh từ các peers trước khi trả về một phản hồi.

Ta có thể dùng lệnh `peer lifecycle chaincode querycommitted` để xác nhận rằng chaincode definition đã được commit lên channel.

In [ ]:
peer lifecycle chaincode querycommitted --channelID mychannel --name basic

Nếu chaincode đã được commit thành công lên channel, lệnh `querycommitted` sẽ trả về **sequence** và **version** của chaincode definition.

In [ ]:
Committed chaincode definition for chaincode 'basic' on channel 'mychannel':
Version: 1.0, Sequence: 1, Endorsement Plugin: escc, Validation Plugin: vscc, Approvals: [Org1MSP: true, Org2MSP: true]

<a id='start-docker'></a>
### 3.6 Start Docker Container

Trong mô hình **CCAAS**, **peers** chỉ là các clients. Để có thể tiếp tục, ta cần chạy docker container. Cụ thể, ta cần chạy hai docker container riêng biệt, mỗi cái tương ứng với một peer sử dụng cùng một docker image đã tạo bên trên nhưng khác tên:
*  Cho Org1: tên `peer0.org1.example.com_basic_ccaas` giống như ta đã khai báo trong file `connection.json` khi tạo chaincode package.
*  Cho Org2: ta đặt tên tương tự `peer0.org2.example.com_basic_ccaas`
`CCAAS_SERVER_PORT` là 9999 như khi ta khai báo trong file `connection.json`.

In [ ]:
docker run --rm -d --name peer0.org1.example.com_basic_ccaas  \
                  --network fabric_test \
                  -e CHAINCODE_SERVER_ADDRESS=0.0.0.0:9999 \
                  -e CHAINCODE_ID=$PACKAGE_ID -e CORE_CHAINCODE_ID_NAME=$PACKAGE_ID \
                    basic_ccaas_image:latest

docker run  --rm -d --name peer0.org2.example.com_basic_ccaas \
                  --network fabric_test \
                  -e CHAINCODE_SERVER_ADDRESS=0.0.0.0:9999 \
                  -e CHAINCODE_ID=$PACKAGE_ID -e CORE_CHAINCODE_ID_NAME=$PACKAGE_ID \
                    basic_ccaas_image:latest

<a id='invoke-chaincode'></a>
### 3.7 Invoke chaincode

Sau khi chaincode definition đã được commit lên channel, chaincode sẽ bắt đầu trên các peers tham gia trong channel nơi chaincode được cài đặt. Chaincode **asset-transfer (basic)** đã sẵn sàng để được gọi bởi các **client applications**. Sử dụng lệnh sau để tạo ra một tập các **assets** khởi đầu trên **ledger**. Lưu ý rằng, lệnh gọi (invoke) cần trỏ đến đủ số lượng **peers** cần thiết để đáp ứng **chính sách phê duyệt**.  

In [ ]:
peer chaincode invoke -o localhost:7050 --ordererTLSHostnameOverride orderer.example.com --tls --cafile "${PWD}/organizations/ordererOrganizations/example.com/orderers/orderer.example.com/msp/tlscacerts/tlsca.example.com-cert.pem" -C mychannel -n basic --peerAddresses localhost:7051 --tlsRootCertFiles "${PWD}/organizations/peerOrganizations/org1.example.com/peers/peer0.org1.example.com/tls/ca.crt" --peerAddresses localhost:9051 --tlsRootCertFiles "${PWD}/organizations/peerOrganizations/org2.example.com/peers/peer0.org2.example.com/tls/ca.crt" -c '{"function":"InitLedger","Args":[]}'

Nếu thành công, ta sẽ thấy output tương tự như sau

In [ ]:
2020-02-12 18:22:20.576 EST [chaincodeCmd] chaincodeInvokeOrQuery -> INFO 001 Chaincode invoke successful. result: status:200

Ta có thể sử dụng một hàm truy vấn để đọc danh sách các **assets** (ở đây là các *cars*) được tạo ra bởi chaincode.

In [ ]:
peer chaincode query -C mychannel -n basic -c '{"Args":["GetAllAssets"]}'

Truy vấn trên có phản hồi là danh sách các assets sau

In [ ]:
[{"Key":"asset1","Record":{"ID":"asset1","color":"blue","size":5,"owner":"Tomoko","appraisedValue":300}},
{"Key":"asset2","Record":{"ID":"asset2","color":"red","size":5,"owner":"Brad","appraisedValue":400}},
{"Key":"asset3","Record":{"ID":"asset3","color":"green","size":10,"owner":"Jin Soo","appraisedValue":500}},
{"Key":"asset4","Record":{"ID":"asset4","color":"yellow","size":10,"owner":"Max","appraisedValue":600}},
{"Key":"asset5","Record":{"ID":"asset5","color":"black","size":15,"owner":"Adriana","appraisedValue":700}},
{"Key":"asset6","Record":{"ID":"asset6","color":"white","size":15,"owner":"Michel","appraisedValue":800}}]

<a id='app'></a>
### 3.8 Sử dụng Application

In [ ]:
cd /home/hyper/go/src/fabric-samples/asset-transfer-basic/application-gateway-javascript/

npm install

node src/app.js

<a id='updrage-chaincode'></a>
### 3.9 Upgrade một Smart Contract

Sau khi một chaincode đã được triển khai trên channel, ta có thể sử dụng cùng một tiến trình `lifecycle chaincode` để upgrade chaincode đó. Các thành viên trên channel có thể upgrade một chaincode bằng cách cài đặt một **chaincode package** mới và phê duyệt một **chaincode definition** với **package ID** mới, một **version** mới, và với tham số **sequence** tăng thêm 1. Chaincode mới có thể được sử dụng sau khi chaincode definition được commit lên channel (với đầy đủ quy trình phê duyệt cần thiết).

Các thành viên channel cũng có thể sử dụng tiến trình upgrade để thay đổi **chính sách phê duyệt**. Bằng cách phê duyệt một chaincode definition với chính sách phê duyệt mới và commit lên channel, các thành viên channel có thể  thay đổi chính sách phê duyệt mà không cần cài đặt một chaincode package mới.

<a id='clean-up'></a>
### 3.10 Dọn dẹp

Sau khi hoàn thành sử dụng chaincode, ta có dọn dẹp bằng cách remove các container CCAAS đã tạo

In [ ]:
docker stop peer0.org1.example.com_basic_ccaas peer0.org2.example.com_basic_ccaas

docker rm peer0.org1.example.com_basic_ccaas peer0.org2.example.com_basic_ccaas

Sau đó ta tắt test network, sử dụng lệnh sau từ thư mục `test-network`.

In [ ]:
./network.sh down

Nếu muốn xóa docker image của chaincode basic_ccaas

In [ ]:
docker rmi basic_ccaas_image:latest

Nếu muốn xóa chaincode package file `basic.tar.gz` (ở thư mục `fabric-samples/asset-transfer-basic/chaincode-external`), ta đi đến thư mục đó và xóa

In [ ]:
rm -f basic.tar.gz

Đi đến thư mục `fabric-samples/asset-transfer-basic/application-gateway-javascript/` và xóa Node.js "groceries"

In [ ]:
rm -rf node_modules package-lock.json